# Numpy  
vectorized operations implemented in C
*(>10x faster)*

In [1]:
import numpy as np
#import numpy and give the variable name np now we can use np to access its functions
print(np.__version__)
#check version of numpy

2.5.0


In [2]:
#without numpy
my_list = [1,2,3,4]
my_list = my_list * 2
print(my_list)
#multiplying a list by 2 will concatanate the list with itself while what we want is to multiply each element by 2

[1, 2, 3, 4, 1, 2, 3, 4]


In [3]:
#now with numpy
arr = np.array([1,2,3,4])
arr = arr * 2
print(arr)
print(type(arr))

[2 4 6 8]
<class 'numpy.ndarray'>


## Multidimensional arrays & indexing

In [4]:
# now arrays have dimensions and shape, dimensions is obviously the number of dimensions or axis it has, while shape is the number
# of elements in each dimension
array_0d = np.array('A') #think single point
print (array_0d.ndim) 
print (array_0d.shape)
#while a 1d:
array_1d = np.array(['A','B','C']) #similar to a line
print(array_1d.ndim)
print(array_1d.shape)
#while a 2d:
array_2d = np.array([['A','B','C'],
                     ['D','E','F']])
print(array_2d.ndim)
print(array_2d.shape)
#while a 3d:
array_3d = np.array([[['A','B','C'],['D','E','F']],
                     [['G','H','I'],['J','K','L']]])
print(array_3d.ndim)
print(array_3d.shape)

0
()
1
(3,)
2
(2, 3)
3
(2, 2, 3)


In [5]:
#however np arrays have to be of the same data type as such np will automatically implicitly upcast all data types to the lowest common denominator that can handle all the data
upcast_array = np.array([1,2.5,5])
print(upcast_array) # notice the dot implying float

[1.  2.5 5. ]


In [6]:
#the axis/dimensions also have to be homogeneous in size so they all have to have the same number of elements in each dimension, otherwise it will be a ragged array and will be treated as an object array
#ragged_array = np.array([[1,2,3],[4,5]])
#print(ragged_array)#Error

In [7]:
#finally accessing elements in a np array is similar to lists where we would use something called chain indexing
#ex:
print(array_2d[0][2])
#however this is not the most efficient or fastest way, as we also have what is called multi-dimensional indexing which is more efficient and faster
print(array_2d[0,2])

C
C


## slicing and dimension collapse

In [8]:
array = np.array([[1,2,3,4]
                  ,[5,6,7,8]
                  ,[9,10,11,12]
                  ,[13,14,15,16]])
#how slicing follows this syntax array[start:end(exclusive):step]
#for example say we want the first row
print(array[0:1])
print()#for clarity
#what if we want the first and second
print(array[0:2])
print()
#what if we want first and third (step of 2)
print(array[0:3:2])
print()
#we can also do the same with columns
print(array[:,0])#notice the first axis has a ':' which numpy interprates as grab all, and it is actually what it autofills if we stop at the first axis
#example print(array[0:2])  actually expands to print(array[0:2,:])
print()
#now if we want the first and third col
print(array[:,0::2])#ommiting the end to mean grab till end


[[1 2 3 4]]

[[1 2 3 4]
 [5 6 7 8]]

[[ 1  2  3  4]
 [ 9 10 11 12]]

[ 1  5  9 13]

[[ 1  3]
 [ 5  7]
 [ 9 11]
 [13 15]]


now notice when we did `print(array[:,0])`  
the output was `[ 1  5  9 13]`
but when when we did `print(array[:,0::2])`
the output became  
```text
[[ 1  3]
[ 5  7]
[ 9 11]
[13 15]]
```  
now why is this important? notice the different shape of each output, yes ofcourse the number of elements is different, duh  
but i am referring to the number of brackets as well, notice that one has one bracket (looks like a 1d vector), while
the other has 2, aka it is a matrix, now we can even prove this by printing their shapes

In [9]:
print(array[:,0].shape)
print(array[:,0::2].shape)

(4,)
(4, 2)


notice how the second shows `(4, 2)` while the first shows `(4,)` not `(4,1)` but `(4,)` that shows that it is indeed collapsed into a 1d vector  
this is because numpy when slicing if we choose a specific index instead of a subsection(even if it had a size of 1) collapses that axis down  
so `array[:,0]` isnt a 2d matrix with 1 column and 4 rows but actually a simply 1d vector  
had we wanted to get the 2d we would use `array[:,0:1]`

In [10]:
print(array[:,0:1].shape)

(4, 1)


this is __very__ important for dimensional mismatch later on, especially with DL and weights.

## Arithmetic  
there are many different kinds of arithmetic supported by numpy but they all depend on how numpy optimizes vector operations
so while a regular scalar operation would, when adding two arrays, load from MEM a[0] then load from MEM b[0] then add then store in c[0] then go to a[1]..etc  
numpy uses many optimizations to cut this down by more than 10x
first it uses the fact that CPUs haver caches which are 100s of times faster to query than the memory (L1,L2,L3 in increasing order of size and decreasing speed), using said caches would be great but regularly python stores each element in a list as a pointer to an object
since elements and usually non homogenous (can mix diff. types), but since numpy enforces strict type homogeniety, now we can load all elements next to each other and load a bunch together into cache, absolutely minimizing cache misses (which makes us query the memory faaaar less)  
second, it makes use of special instruction set extensions called AVX2 and AVX-512(for intel processors, AMD has similar extenstions), what it basically does is make use of a hardware component, that is expanding the 64/128 bit registers in place to accomodate 256 and 512 bits respectively (called YMM and ZMM registers), and a software component, that is new Floating point instructions specifically made to use the new larger registers to make use of them to calculate many floating point operations at once, think for example if a single element was 32 bits, a single YMM register could accomodate `256/32 = 38` elements if we were to use these to apply a single operation to all of them at the same time (SIMD) we would cut running time by 8x
thirdly, it also uses many other optimizations such as highly optimized external math libraries (BLAS, intel MKL, etc) that allow multithreading (remember python GIL strictly enforces one thread to run python code at a time, bottlenecking multicore usage), but we won't go into much details on the other optimizations for now.

### Scalar arithmetic

scalar as in adding a singular element to a whole array, this uses a concept called broadcasting that we will go into in detail later, for now it is enough to know that it means *stretching* the element/smaller array to the size of the bigger one so operations are viable.
so when lets say we want to add `[1,2,3] + 3`, regular python lists would throw an error while numpy would first duplicate the element and add it to all elements in our bigger array, `[1,2,3] + [3,3,3]` and making use of our previously explained optimization cut down on running time significantly without a single for loop in sight! 

In [11]:
arr = np.array([1,2,3])
print(arr + 3)

[4 5 6]


### Vectorized math functions

simply an extension of the previous idea, where now we can use built-in numpy math functions and constants on all array elements at once!

In [12]:
print(np.sqrt(arr))

tmp_float_arr = np.array([1.3,2.7,3.9])
print(np.round(tmp_float_arr))

print(np.pi)

#calculate circle areas for radii
radii = np.array([1,5,20,100])
print(np.pi * radii ** 2) #done in 2 cycles (not accounting for time to load in cache) instead of 4 *4 * 4 ~= 256

[1.         1.41421356 1.73205081]
[1. 3. 4.]
3.141592653589793
[3.14159265e+00 7.85398163e+01 1.25663706e+03 3.14159265e+04]


### Element-wise arithmetic  

doing arithmetic between two arrays element-by-element (again utilizes all our previous optimizations) and follows broadcasting and shape matching rules, we will discuss those later.

In [13]:
arr1 = np.array([[1,2,3],[4,5,6],[7,8,9]])
arr2 = np.array([[9,8,7],[6,5,4],[3,2,1]])
print(arr1 + arr2) #can also use np.add(arr1,arr2) but both are congruent EXCEPT when you want to use np.add options like out,..,etc

[[10 10 10]
 [10 10 10]
 [10 10 10]]


### Comparison operators

Now, these are an actually fascination usage of numpy think if we wanted to check an array of scores for passing scores, and return a true/false depending on each score or maybe even assign a new values for failing scores, regular python would do something like this:  
```python
scores = [30,60,78,100]
for x in scores:
    if (x > 60):
        print(x) #/true
    else:
        print(0) #/false
```
however in numpy this is all that takes

In [14]:
scores = np.array([30,60,78,100])
scores[scores < 60] = 0
print(scores)

[  0  60  78 100]


and ofcourse the running time is magnitudes faster

this syntax `arr[arr > x]` is actually called a boolean mask where numpy does its optimized SIMD operations to calculate a boolean mask for the array to evaluate the operation on each element returning something that looks like `[true, false, false, false]` it then applies that mask likes a physical shield to the cache only allowing the 0 to pass to locations that don't have a false to physically protect it

## Broadcasting

so, broadcasting is this amazing virtual(doesnt actually copy data) that numpy does to make matrixx operations (addition, subtraction,hadamard mul, matmul (only on batch dimensions but more on that later),etc) viable between matrices/array that have different dimensions and size of said dimensions  
now the trick is simple if there is an operation between two arrays, such that they have different sized dimensions we virtually expand the smaller dimension each time by copying the lower dimension element and reusing it AS IF we had a full axis down that data lane, notice how i spoke using singular terms, that is because numpy can only broadcast, that is expand, an array if it has only one element in the expanding axis that is it has a 1 in it's shape there, commonly referred to as a singleton axis, now as to why that is, it comes down to non-ambiguity, if you had one element a (4) and wanted to make a 4 element array of it what can you do? only one thing, that is copy it 4 times into `[4,4,4,4]` but what about if you had [4,5] and you want 4 elements? well one option is `[4,4,5,5]` another is `[4,5,4,5]`; numpy decided to avoid said non-ambiguity and simply throws a `ValueError` if you try such a thing.  
circling back, this is how broadcasting works, it compares each dimension for both arrays right-to-left prepending 1s to the smaller (dimension wise) one till they are the same shape, it allows operations in two cases, either the dimension matches up, in which case we need not do anything, yay! or one of them has a 1 in which case we copy each element in the singleton axis down that axis till it matches the other one.
Ex: assume we had two matrices `(4,4)` and `(4,1)` if we were to add them, numpy would compare them as such  
(4,4)  
(4,1)  
going right to left we have a 4 and 1, a mismatch but one is a singleton axis so it would copy every element down that axis 4 times(VIRTUALLY, this is very important),so now the operation is between a `(4,4)` and `(4,4)` conceptually, then next axis we have a 4,4 which matches and the operation is done.
another Ex is if we had a `(4,4)` and a `(4,)` or a less dimensional array, as we said numpy would prepend axis of size 1 onto it from the *LEFT* turning it into `(1,4)` and do the same as last example. 

In [15]:
#live exampe
matA = np.array([[1,2,3,4]
                 ,[5,6,7,8]
                 ,[9,10,11,12]
                 ,[13,14,15,16]])
matB = np.array([[1]
                 ,[2]
                 ,[3]
                 ,[4]])
print(matA.shape)
print(matB.shape)
print(matA + matB)
print((matA + matB).shape)

print()
matC = np.array([1,2,3,4])
print(matC.shape)
print(matA + matC)
print((matA + matC).shape)

(4, 4)
(4, 1)
[[ 2  3  4  5]
 [ 7  8  9 10]
 [12 13 14 15]
 [17 18 19 20]]
(4, 4)

(4,)
[[ 2  4  6  8]
 [ 6  8 10 12]
 [10 12 14 16]
 [14 16 18 20]]
(4, 4)


you will notice while both outputs have the same shape `(4,4)` they have different values, even though the underlying arrays, had the same elements, but they were arranged differently (4 element column vs 4 element row) which fundamentally changed how they got broadcasted and added.

## Aggregate functions

as the name implies, aggregate functions *aggregate* a specific function across our whole array or across a specific dimension as specified.  
we will be discussing the most used/useful ones along with certain use cases and tips/tricks/edge cases.

however, before that we must specifiy that to use aggregate functions across a specific dimension we simply specify the parameter axis=(axis number), where setting axis = k destroys/compresses the kth dimension(0-based), for example setting axis = 1 turns a matrix with the shape `(A, B, C, D)` into `(A, C, D)`

also, it is often useful to set `keepdims = true` if we plan to use the output for broadcasting, as remember that in broadcasting numpy always prepends 1s to the left as such using some function with an array of shape `(5,3)` and `axis = 1` will lead to shape `(5,)` output which when broadcasted transforms to `(1,5)` completely messing up the original shape and giving a swift `ValueError`. 

for future examples we shall use the following array as an example:

In [16]:
arr = np.array([[[1,2,3],[4,5,6],[7,8,9]]
                ,[[10,11,12],[13,14,15],[16,17,18]]
                ,[[19,20,21],[22,23,24],[25,26,27]]])
print(arr.shape)

(3, 3, 3)


### `np.sum` and `np.mean`

definitely the most used and famous functions, as the name implies, they respectively return the sum and mean of our array/axis  
ex:

In [17]:
tmp = np.mean(arr)#across whole array
print(tmp)
print(tmp.shape)#notice shape when no axis is defined
tmp = np.mean(arr,axis=2)#across columns
print(tmp)
print(tmp.shape)#notice how the output completely LOST that dimension as such think how it would be prepended to broadcast? ans(1,3,3) while desired to reuse with originial is (3,3,1)  
#technically since we created the array with all dims at size 3 but imagine an array with size (3,4,5), such an operation would lead to (1,3,4) instead of desired (3,4,1)
tmp = np.mean(arr,axis=2,keepdims=True)
print(tmp)
print(tmp.shape)#contrast with previous shape/output

14.0
()
[[ 2.  5.  8.]
 [11. 14. 17.]
 [20. 23. 26.]]
(3, 3)
[[[ 2.]
  [ 5.]
  [ 8.]]

 [[11.]
  [14.]
  [17.]]

 [[20.]
  [23.]
  [26.]]]
(3, 3, 1)


a quick tip for `np.sum` is that it can often overflow or lose float precision due to truncation at higher values, so it is smart to use `dtype=float64` then cast back  
ex:

In [18]:
arr_32 = np.array([1e7, 1.23456789, -1e7], dtype=np.float32)
bad_sum = np.sum(arr_32)
print(bad_sum)

good_sum = np.sum(arr_32,dtype=np.float64).astype(np.float32)
print(good_sum)

1.0
1.2345679


### `np.var` and `np.std`

standard statistical functions often used for normalization, notice however two important tips:  
1. `np.var` and `np.std` don't use by default **bessel's correction**, in other words when calculating both Std and Var they divide by `N` and not by `N-1`, this can be fixed by setting the degrees of freedom to 1 or `ddof=1`  
2. neither the std nor the var use any epsilon when calculating, as such there are cases when the array is completely equal that var and std will equal 0 which might lead to division by zero erros (think normalization)

In [19]:
print(np.var(arr))
print(np.std(arr,axis=2))

60.666666666666664
[[0.81649658 0.81649658 0.81649658]
 [0.81649658 0.81649658 0.81649658]
 [0.81649658 0.81649658 0.81649658]]


### `np.max` and `np.min`

not much to say here, just the regular max and min functions

In [20]:
print(np.max(arr,axis = 0))
print(np.min(arr,axis = 1))

[[19 20 21]
 [22 23 24]
 [25 26 27]]
[[ 1  2  3]
 [10 11 12]
 [19 20 21]]


### `np.argmax` and `np.argmin`

more interesting that the previous two, these functions don't return the max/min rather their index in the array, which is often just as if not more useful than the value itself, think if you had a model that predicted through a softmax a certain class, rather than wasting time checking each class adding a constant time loss `np.argmax` can solve that far faster, as for `np.argmin` a famous use would centroid calculations in clustering (more on that in the unsupervised learning section)  
An important tip for using either however, is that they return a scalar value which if the array shape is not 1D makes it often hard to pinpoint the actual positing of the min/max, numpy thankfully offers a function called `unravel_index(output, array.shape)`

In [21]:
print(np.argmax(arr))
print(np.unravel_index(np.argmax(arr),arr.shape))#notice the output is wrapped in np.int64, important for later usage

26
(np.int64(2), np.int64(2), np.int64(2))


### `np.any` and `np.all`

succinctly defined as an OR and a AND operator respectively, that is to say, they are used to find if any/all value in the array/axis satisfies a certian condition, more on that in the examples, however before we get to that it would be helpful to learn about two *element-wise* functions:


#### `np.isnan` and `np.isinf`

as the name implies they are used to check element-wise whether the element is nan or infinity respectively, the output is a boolean mask.

In [22]:
#quick example of np.isnan before we get to any/all
print(np.isnan(arr))


[[[False False False]
  [False False False]
  [False False False]]

 [[False False False]
  [False False False]
  [False False False]]

 [[False False False]
  [False False False]
  [False False False]]]


In [23]:

#to use np.any and np.all simply pass a boolean mask into them and they will return a true or false aggregated across it
#for example lets say we want to find whether any row(across columns) contains a value that is greater than the mean of the array
mean = np.mean(arr)
mask = arr > mean
print(mask)
print()#newline for clarity
print(np.any(mask,axis=2))
print()
#what if we want to check if ALL elements of a row are greater
print(np.all(mask,axis=2))
print()

[[[False False False]
  [False False False]
  [False False False]]

 [[False False False]
  [False False  True]
  [ True  True  True]]

 [[ True  True  True]
  [ True  True  True]
  [ True  True  True]]]

[[False False False]
 [False  True  True]
 [ True  True  True]]

[[False False False]
 [False False  True]
 [ True  True  True]]



## Filtering

so filtering is the act of selecting/dropping certain elements from an array based on a condition(s), it almost always includes the formation of a boolean mask(even if implicitly) based on a condition/comparison operator, which is then used to filter the array like stencil (Frue values pass, False is blocked)  
note also that conditions can be used with logical operators to create complex conditions, however you must use bitwise operators (~,&,|) as the python standard (and,or,not) compare the object as a whole and not the elementwise masks

In [ ]:
#to create a boolean mask, you use the name of the array (can be augmented with selecting an axis to compare on)
arr = np.array([0,-1,2,5,20,3])
print(arr < 0) 
#to then use the boolean mask to filter the array you simply pass it inside the subscript operator of the array
print(arr[arr < 0]) 
print((arr > 0) & (arr < 20)) #notice the use of brackets as logical bitwise operators take mathematical precedence over other operations

[False  True False False False False]
[-1]
[False False  True  True False  True]


two interesting things to know are that comparisons are vectorized, so they are optimized same as other operations, and that using boolean masks only for filtering leads to flattening the output into a 1d array, completely messing up the shape of the array:

In [30]:
arr = np.array([[0,-1,2,4,5]
                ,[100,2,-3,1,6]])
mask = arr < 0
print(mask)
print(arr[mask])
print(arr[mask].shape)

[[False  True False False False]
 [False False  True False False]]
[-1 -3]
(2,)


this happens because numpy doesn't if the output can be made into the same shape of the input (more often it can't if you remove even one element), as such it flattens the array, this can be remedied by instead changing the values of false and/or true elements and returning the same array shape, this can be done in two ways  
1. `np.where(mask, value of true, value of false)`
2. in-place mutation

In [32]:
#using arr from the previous cell
tmp = np.where((arr > 0) & (arr < 20),arr,-100)
print(tmp)

[-100 -100    2    5 -100    3]


notice however, that `np.where` has a big glaring downside, it ALWAYS allocates a completely new array in memory, which if you were dealing with huge weights matrices can crash your work.

the second option is inplace-mutation, which while it, as it's name suggests, mutates array elements in-place :  
syntax: `arr[mask] = value` will set true indices to `value`  
it can't however do two step changes in one line like np.where, and can be prone to error if done wrongly.
lets say for example's sake, we had an array we wanted to set elements that are less than 0 to 99 and positive elements to 100 : 

In [ ]:
arr = np.array([-2,100,99])
arr[arr <= 0] = 99
arr[arr != 99] = 100
print(arr) # the right output is [99,100,100]

[ 99 100  99]


notice how since we changed the ground truth itself, it has become hard to mutate the array properly especially if it already contains the value we chose, which with big enough arrays is quite probable.  
the solution is simple, just create the mask **before** you mutate the array and use it twice

In [35]:
arr = np.array([-2,100,99])

mask = arr <= 0
arr[mask] = 99
arr[~mask] = 100
print(arr)

[ 99 100 100]


you will notice that this did indeed create another array in memory, however masks are internally stored as 1 byte per element, compared to 8 bytes (atleast for a float32) that are reserved by `np.where`, this is an absolute win memory-wise.

another very useful feature of filtering is the ability to drop rows/columns(axis in general) by using a mask that matches that dimensions size
lets say we had an array of patients(patient id,age,income) and we wanted to drop patients (rows) that had a low income (< 20), lets watch how we can do this:

In [39]:
patients = np.array([[0,52,100]
                     ,[1,18,10]
                     ,[2,30,200]])
mask = patients[:,2] < 20 #we use :,2 to say for all rows (:) and (,) for column 2 (2)
print(mask) #notice this evaluated as true for the row we want to drop
#we can then reuse the mask in the same way to do this
print(patients[~mask]) #notice we used ~ as we want the rows that DIDN'T pass the condition and also we used mask in the first part of the..
#subscript operator so we use it on the first dimension (rows), this is good to know when you have multi-dimensional arrays Ex : (batch, channel, row, col)

[False  True False]
[[  0  52 100]
 [  2  30 200]]
